# Statim: big synthetic-data run on an A100 (auto-upload)

**Before you start**
1. *Runtime → Change runtime type → A100 GPU → Save.*
2. Hugging Face token for the upload: on huggingface.co → Settings → Access Tokens → *New token* (type **Write**).
   In Colab, click the **key icon** (Secrets) in the left sidebar → *Add new secret* → name `HF_TOKEN`, paste the
   token, switch **Notebook access** on. The token never appears in this notebook.

**Then: Runtime → Run all (Ctrl+F9).** The second one runs for hours and shows live progress. It uploads to the **private**
dataset `Beko2210/statim-synth-staging` after every stage, so nothing is lost if Colab disconnects: run the cell
again and it resumes.

Writer: Qwen3-30B-A3B-Instruct-2507 (Apache-2.0). Checker: Mistral-Small-3.2-24B (Apache-2.0, multilingual, blind);
second checker for the European languages: Phi-4 (MIT). A row is kept only when every checker agrees.
Languages without a seed corpus (tr, ja, zh, hi, ru) are written from English passages.
Seeds: BillSum (CC0), GovReport (CC-BY-4.0), EUR-Lex-Sum (CC-BY-4.0). The output is *unchecked* for
evaluation overlap until pop-os runs `tools/synth/leakage.py --filter`; nothing here is published.


In [ ]:
# ===== SETUP (~8 min) =====
import os, signal, subprocess, sys, time, json, urllib.request
# round1 (2026-10-04): urgency,nli,spam,sarcasm,emotion,claim,stance,reading at 4000 each
# round2 (2026-10-04): factcheck,sentiment,emotion,injection at 6000 each
TASKS = "emotion,sentiment,pii,formality"
TASK_LANGS = "emotion=tr/ja/zh/hi/ru,sentiment=tr/ja/zh/hi/ru"   # the new gate languages only
PER_CAPABILITY = 4000        # kept items per capability before the second check; 4 x 4000 = 16,000
ROUND = "round3"             # a new name starts a new batch; the same name resumes
STAGING_REPO = "Beko2210/statim-synth-staging"   # private
GENERATOR = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"
FALLBACK_GENERATOR = "Qwen/Qwen3-30B-A3B-GPTQ-Int4"
VERIFIER = "RedHatAI/Mistral-Small-3.2-24B-Instruct-2506-FP8"   # all languages
VERIFIER2 = "microsoft/phi-4"                                       # second check, European languages
RECHECK_LANGS = "en,de,fr,es,it,pt,nl,pl"
VLLM_ARGS = {VERIFIER: "--tokenizer_mode mistral --config_format mistral --load_format mistral"}
VLLM_VERSION = "0.30.0"

def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("FAILED: " + cmd)
    return r.stdout

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     text=True, capture_output=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU. Runtime > Change runtime type > A100 GPU.")
print("GPU:", gpu)

from google.colab import userdata
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    raise SystemExit("Add the secret HF_TOKEN (key icon in the left sidebar) and allow notebook access.")
os.environ["HF_TOKEN"] = HF_TOKEN

sh(f"pip -q install vllm=={VLLM_VERSION} huggingface_hub datasets pyarrow")
os.chdir("/content")
sh("rm -rf /content/statim && git clone -q --depth 1 -b data/synth-v2 https://github.com/BEKO2210/statim /content/statim")
os.chdir("/content/statim")
print(sh("git log --oneline -1"))

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_repo(STAGING_REPO, repo_type="dataset", private=True, exist_ok=True)
print("staging repo (private):", STAGING_REPO, "| logged in as", api.whoami()["name"])
REV = {m: api.model_info(m).sha for m in (GENERATOR, FALLBACK_GENERATOR, VERIFIER, VERIFIER2)}
print("revisions:", REV)
print("SETUP OK")

In [ ]:
# ===== BIG RUN (hours; live progress; resumes; uploads every 15 min and after every stage) =====
import re
WORK = f"/content/{ROUND}"
CANDS, OUT = f"{WORK}/candidates.jsonl.gz", f"{WORK}/out"
os.makedirs(WORK, exist_ok=True)

# resume from the private staging repo if this round was started in an earlier session
try:
    from huggingface_hub import snapshot_download
    snapshot_download(STAGING_REPO, repo_type="dataset", token=HF_TOKEN, local_dir="/content",
                      allow_patterns=[f"{ROUND}/*"])
    print("resumed files:", sorted(os.listdir(WORK)))
except Exception as exc:
    print("nothing to resume yet:", type(exc).__name__)

def upload(stage):
    api.upload_folder(folder_path=WORK, path_in_repo=ROUND, repo_id=STAGING_REPO, repo_type="dataset",
                      commit_message=f"{ROUND}: {stage}")
    print(f"uploaded {ROUND} after {stage} -> {STAGING_REPO} (private)", flush=True)

import threading
_stop_sync = threading.Event()

def _periodic_upload(every=900):
    """Upload the round every 15 minutes while a stage runs: a disconnect loses at most that much."""
    while not _stop_sync.wait(every):
        try:
            upload("periodic")
        except Exception as exc:
            print("periodic upload failed (will retry):", type(exc).__name__, flush=True)

threading.Thread(target=_periodic_upload, daemon=True).start()

def start_vllm(model, revision):
    log = open(f"/tmp/vllm-{model.split('/')[-1]}.log", "w")
    proc = subprocess.Popen(f"exec vllm serve {model} --revision {revision} --port 8000 --max-model-len 4096 "
                            f"--gpu-memory-utilization 0.90 --enable-prefix-caching --max-num-seqs 256 "
                            f"{VLLM_ARGS.get(model, '')}",
                            shell=True, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    t0, last = time.time(), -30.0
    while True:
        try:
            body = json.dumps({"model": model, "max_tokens": 5, "messages": [{"role": "user", "content": "Say ok"}]}).encode()
            urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions", data=body,
                                   headers={"Content-Type": "application/json"}), timeout=60).read()
            print(f"vLLM ready with {model} after {time.time() - t0:.0f} s", flush=True)
            return proc
        except Exception:
            if proc.poll() is not None:
                print(open(log.name).read()[-3000:])
                raise RuntimeError("vLLM stopped while loading " + model)
            if time.time() - last >= 30:
                last = time.time(); print(f"  loading {model} ... {time.time() - t0:.0f} s", flush=True)
            time.sleep(2)

def stop_vllm(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM); proc.wait(timeout=90)
    except Exception:
        try: os.killpg(proc.pid, signal.SIGKILL)
        except Exception: pass
    subprocess.run("pkill -f 'vllm serve'; sleep 8", shell=True)
    print("GPU memory after stop:", sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader").strip())

def live(args):
    """Run grounded.py and stream its output line by line; return (exit code, last 40 lines)."""
    cmd = [sys.executable, "-u", "tools/synth/grounded.py", *args, "--cache", "/tmp/statim-synth-corpora"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=dict(os.environ, TMPDIR="/tmp"))
    tail = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    return proc.wait(), "".join(tail)

generator, gen_rev = GENERATOR, REV[GENERATOR]
oversample, write_tasks = 4, TASKS
for attempt in range(1, 4):
    print(f"\n========== {ROUND}: write {write_tasks} (attempt {attempt}, oversample {oversample}) ==========")
    try:
        gen = start_vllm(generator, gen_rev)
    except RuntimeError as exc:
        print(exc, "-> fallback", FALLBACK_GENERATOR)
        subprocess.run("pkill -f 'vllm serve'; sleep 8", shell=True)
        generator, gen_rev = FALLBACK_GENERATOR, REV[FALLBACK_GENERATOR]
        gen = start_vllm(generator, gen_rev)
    code, _ = live(["--stage", "generate", "--backend", "openai", "--host", "http://127.0.0.1:8000",
                    "--generator-model", generator, "--generator-revision", gen_rev,
                    "--per-capability", str(PER_CAPABILITY), "--tasks", write_tasks, "--task-langs", TASK_LANGS, "--concurrency", "128",
                    "--oversample", str(oversample), "--max-jobs-factor", "80", "--candidates", CANDS])
    stop_vllm(gen)
    upload(f"write attempt {attempt}")
    print(f"\n========== {ROUND}: check (attempt {attempt}) ==========")
    ver = start_vllm(VERIFIER, REV[VERIFIER])
    code, tail = live(["--stage", "verify", "--backend", "openai", "--host", "http://127.0.0.1:8000",
                       "--verifier-model", VERIFIER, "--verifier-revision", REV[VERIFIER],
                       "--per-capability", str(PER_CAPABILITY), "--tasks", TASKS, "--task-langs", TASK_LANGS, "--concurrency", "128",
                       "--candidates", CANDS, "--out-dir", OUT, "--defer-leakage-check", "--resume"])
    stop_vllm(ver)
    upload(f"check attempt {attempt}")
    if code == 0:
        print(f"\n{ROUND} COMPLETE: every capability filled.")
        break
    found = re.search(r"SHORT_TASKS: ([a-z_,]+)", tail)
    if not found:
        print(tail[-2000:]); raise SystemExit("check stage failed (see above)")
    write_tasks = found.group(1)
    print("short:", write_tasks, "-> writing more candidates for these only")
    oversample += 3
else:
    print("\nStill short after 3 attempts; the uploaded data is complete for every filled capability.")

print(f"\n========== {ROUND}: second check ({VERIFIER2}, {RECHECK_LANGS}) ==========")
ver2 = start_vllm(VERIFIER2, REV[VERIFIER2])
code, tail = live(["--stage", "recheck", "--backend", "openai", "--host", "http://127.0.0.1:8000",
                   "--verifier-model", VERIFIER2, "--verifier-revision", REV[VERIFIER2],
                   "--tasks", TASKS, "--task-langs", TASK_LANGS, "--concurrency", "128",
                   "--in-dir", OUT, "--out-dir", f"{WORK}/final", "--recheck-langs", RECHECK_LANGS])
stop_vllm(ver2)
upload("second check")
if code:
    print(tail[-2000:]); raise SystemExit("second check failed (see above)")
OUT = f"{WORK}/final"

import pandas as pd
m = json.load(open(f"{OUT}/manifest.json"))
display(pd.DataFrame([{"capability": t, "kept": s["kept"], "rechecked": s["rechecked"],
                       "second checker agrees": "%.0f%%" % (100 * s["agreement"]) if s["agreement"] is not None else "-"}
                      for t, s in m["recheck"]["tasks"].items()]))
